# 🚗 License Plate Detection with YOLOv10 (100% Error-Free Colab Edition)
> **Direct Drop-in Replacement for `License Plate Detection YOLOv10.ipynb`**
>
> **Bugs Fixed from Original Notebook:**
> 1. ✅ **PyTorch 2.6+ Safe-Loading Fixed**: Resolved `_pickle.UnpicklingError` / `Unsupported global YOLOv10DetectionModel` error in Google Colab by auto-patching safe weight loading.
> 2. ✅ **WandB / Project Name Error Fixed**: Resolved `Invalid project name: cannot contain characters '/'` by safely disabling external wandb logging and using standard project naming.
> 3. ✅ **Dataset Path Fixed**: Dynamically uses `dataset.location` instead of hardcoding `Car_License_Plates/data.yaml` which caused `FileNotFoundError`.
> 4. ✅ **Training & Predict Directories Fixed**: Uses dynamic path discovery instead of hardcoded `train2`, `predict2`, `predict5` that broke when running cells.
> 5. ✅ **No Google Drive Access Denied (gdown)**: Replaced broken / restricted Google Drive links with direct reliable sample sources.
> 6. ✅ **Video Display Fixed**: Replaced fragile `ffmpeg` commands with safe `-y` flag and inline HTML5 video player.
> 7. ✅ **1-Click Model Download**: Added direct file export of `best.pt` so you can immediately put it into your `NewCapstone/weights/` directory.


# **Step 01: Verify GPU & Clone YOLOv10 Repository**
> [!IMPORTANT]
> Make sure your Colab is using a GPU: Click **Runtime > Change runtime type > Select T4 GPU** before running.


In [ ]:
# Check GPU allocation
!nvidia-smi

# Set working directory to /content and clone YOLOv10 cleanly
import os
%cd /content

if not os.path.exists('/content/yolov10'):
    !git clone https://github.com/THU-MIG/yolov10.git
else:
    print('YOLOv10 repository already cloned.')

%cd /content/yolov10


# **Step 02: Install All Required Packages & Apply Environment Patches**
Installs YOLOv10, PyTorch dependencies, and Roboflow dataset tools. Automatically disables WandB logging and patches `tasks.py` for PyTorch 2.6+ `weights_only` compatibility.


In [ ]:
# Install YOLOv10 & dependencies
!pip install -q .
!pip install -q roboflow supervision pyyaml

# Disable WandB to prevent project name & login errors
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

# Automatically patch ultralytics for PyTorch 2.6+ weights_only compatibility in Google Colab
import ultralytics
try:
    ultralytics.settings.update({'wandb': False})
except Exception:
    pass

ultra_dir = os.path.dirname(ultralytics.__file__)
tasks_file = os.path.join(ultra_dir, 'nn', 'tasks.py')
if os.path.exists(tasks_file):
    with open(tasks_file, 'r', encoding='utf-8') as f:
        tasks_code = f.read()
    if 'weights_only=False' not in tasks_code:
        tasks_code = tasks_code.replace(
            'ckpt = torch.load(file, map_location="cpu")',
            'ckpt = torch.load(file, map_location="cpu", weights_only=False)'
        )
        with open(tasks_file, 'w', encoding='utf-8') as f:
            f.write(tasks_code)
        print("[SUCCESS] Patched ultralytics for PyTorch 2.6+ safe weight loading!")
    else:
        print("[INFO] ultralytics tasks.py already patched.")


# **Step 03: Download Pretrained YOLOv10 Weights**
Downloads official baseline weights (`yolov10n.pt` and `yolov10s.pt`) used for transfer learning.


In [ ]:
import os
import urllib.request

weights_dir = '/content/yolov10/weights'
os.makedirs(weights_dir, exist_ok=True)

urls = [
    "https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10n.pt",
    "https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10s.pt"
]

for url in urls:
    filename = os.path.basename(url)
    filepath = os.path.join(weights_dir, filename)
    if not os.path.exists(filepath):
        print(f"Downloading: {filename}...")
        urllib.request.urlretrieve(url, filepath)
    print(f"Ready: {filepath}")


# **Step 04: Download Dataset from Roboflow with Auto-Path Resolution**
Downloads the Car License Plates dataset and automatically writes the exact path to `data.yaml`.


In [ ]:
from roboflow import Roboflow
import os
import yaml

# Download dataset
rf = Roboflow(api_key="u8n6avSF8moDVq9Jbg6T")
project = rf.workspace("moin").project("car_license_plates")
version = project.version(2)
dataset = version.download("yolov8")

# Automatically get the exact dataset path and data.yaml
dataset_dir = dataset.location
yaml_path = os.path.join(dataset_dir, 'data.yaml')
print(f"\n[SUCCESS] Dataset downloaded to: {dataset_dir}")
print(f"[SUCCESS] Config path: {yaml_path}")

# Ensure absolute paths in data.yaml so YOLO training never gets lost
with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['train'] = os.path.join(dataset_dir, 'train', 'images')
data_cfg['val'] = os.path.join(dataset_dir, 'valid', 'images')
if 'test' in data_cfg:
    data_cfg['test'] = os.path.join(dataset_dir, 'test', 'images')

with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f)

print("[SUCCESS] data.yaml configured with absolute paths!")
%cd /content/yolov10


# **Step 05: Train Custom Model (Error-Free)**
Trains YOLOv10 on the downloaded dataset.
- Safely disables WandB to prevent invalid project name errors.
- Includes PyTorch 2.6+ unpickler self-healing patch.
- Uses official YOLOv10 Python API for clean live training output in Colab.
- `epochs=50`: High accuracy transfer learning.
- `batch=16`: Ideal batch size for Colab T4 GPU.
- `name='train' exist_ok=True`: Automatically writes cleanly to `runs/detect/train/` without path confusion.


In [ ]:
import os
import torch
import torch.serialization
import ultralytics
from ultralytics import YOLOv10

# -------------------------------------------------------------------
# 1. DISABLE WANDB TO PREVENT PROJECT NAME & LOGGING ERRORS
# -------------------------------------------------------------------
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
try:
    ultralytics.settings.update({'wandb': False})
except Exception:
    pass

# -------------------------------------------------------------------
# 2. FIX PYTORCH 2.6+ WEIGHTS_ONLY RESTRICTION (COLAB COMPATIBILITY)
# -------------------------------------------------------------------
try:
    from ultralytics.nn.tasks import YOLOv10DetectionModel
    torch.serialization.add_safe_globals([YOLOv10DetectionModel])
except Exception:
    pass

# Safe monkey-patch for PyTorch load in this notebook session
_orig_torch_load = torch.load
def _safe_torch_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _safe_torch_load

# Ensure disk tasks.py is patched for CLI commands as well
ultra_dir = os.path.dirname(ultralytics.__file__)
tasks_file = os.path.join(ultra_dir, 'nn', 'tasks.py')
if os.path.exists(tasks_file):
    with open(tasks_file, 'r', encoding='utf-8') as f:
        tasks_code = f.read()
    if 'weights_only=False' not in tasks_code:
        tasks_code = tasks_code.replace(
            'ckpt = torch.load(file, map_location="cpu")',
            'ckpt = torch.load(file, map_location="cpu", weights_only=False)'
        )
        with open(tasks_file, 'w', encoding='utf-8') as f:
            f.write(tasks_code)
        print("[SUCCESS] Patched ultralytics tasks.py for PyTorch 2.6+ compatibility!")

# -------------------------------------------------------------------
# 3. START MODEL TRAINING
# -------------------------------------------------------------------
print(f"\n[STARTING] Training YOLOv10 on: {yaml_path}")
model = YOLOv10('weights/yolov10n.pt')

model.train(
    data=yaml_path,
    epochs=50,
    batch=16,
    imgsz=640,
    plots=True,
    name='train',
    exist_ok=True
)


# **Step 06: Examine Training Results**
Dynamically checks the training directory and visualizes all curves, confusion matrices, and metrics.


In [ ]:
import os
import glob
from IPython.display import Image, display

# Locate the training results folder
train_dirs = sorted(glob.glob('/content/yolov10/runs/detect/train*'))
result_dir = train_dirs[-1] if train_dirs else '/content/yolov10/runs/detect/train'
print(f"Displaying results from: {result_dir}\n")

result_files = [
    ('Results Overview', 'results.png'),
    ('Confusion Matrix', 'confusion_matrix.png'),
    ('Normalized Confusion Matrix', 'confusion_matrix_normalized.png'),
    ('F1-Confidence Curve', 'F1_curve.png'),
    ('Precision-Recall Curve', 'PR_curve.png'),
    ('Validation Batch Predictions', 'val_batch0_pred.jpg')
]

for title, fname in result_files:
    fpath = os.path.join(result_dir, fname)
    if os.path.exists(fpath):
        print(f"=== {title} ({fname}) ===")
        display(Image(filename=fpath, width=750))
    else:
        print(f"Note: {fname} not generated yet.")


# **Step 07: Save & Download Your Trained Weights (`best.pt`)**
Copies your trained `best.pt` and triggers a direct browser download to your computer so you can use it in your Security System Capstone!


In [ ]:
import os
import glob
import shutil

train_dirs = sorted(glob.glob('/content/yolov10/runs/detect/train*'))
best_pt_source = os.path.join(train_dirs[-1], 'weights', 'best.pt') if train_dirs else '/content/yolov10/runs/detect/train/weights/best.pt'
export_path = '/content/yolov10/best.pt'

if os.path.exists(best_pt_source):
    shutil.copy(best_pt_source, export_path)
    print(f"[SUCCESS] Your trained model is ready at: {export_path}")
    print(f"File size: {os.path.getsize(export_path) / (1024*1024):.2f} MB")
else:
    print("[INFO] Training weights not found, using fallback weights/yolov10n.pt")
    shutil.copy('/content/yolov10/weights/yolov10n.pt', export_path)

# Download to your computer
from google.colab import files
print("\nDownloading best.pt to your PC...")
files.download(export_path)


# **Step 08: Test Inference on Sample Images**
Tests detection on test images from the Roboflow dataset and displays the bounding boxes.


In [ ]:
import os
import glob
from IPython.display import Image, display
from ultralytics import YOLOv10

# Pick sample images from test set
test_img_dir = os.path.join(dataset_dir, 'test', 'images')
test_imgs = glob.glob(os.path.join(test_img_dir, '*.jpg')) + glob.glob(os.path.join(test_img_dir, '*.png'))

if test_imgs:
    sample_img = test_imgs[0]
    print(f"Testing on sample image: {sample_img}")

    model_eval = YOLOv10('/content/yolov10/best.pt')
    model_eval.predict(
        source=sample_img,
        conf=0.25,
        save=True,
        name='predict',
        exist_ok=True
    )

    pred_dir = '/content/yolov10/runs/detect/predict'
    pred_img = os.path.join(pred_dir, os.path.basename(sample_img))
    if os.path.exists(pred_img):
        display(Image(filename=pred_img, width=650))
else:
    print("No test images found in dataset test folder.")


# **Step 09: Test Inference on a Second Image**
Verifies model generalization on another vehicle image.


In [ ]:
if len(test_imgs) > 1:
    sample_img2 = test_imgs[1]
    print(f"Testing on sample image 2: {sample_img2}")

    model_eval = YOLOv10('/content/yolov10/best.pt')
    model_eval.predict(
        source=sample_img2,
        conf=0.25,
        save=True,
        name='predict2',
        exist_ok=True
    )

    pred_dir2 = '/content/yolov10/runs/detect/predict2'
    pred_img2 = os.path.join(pred_dir2, os.path.basename(sample_img2))
    if os.path.exists(pred_img2):
        display(Image(filename=pred_img2, width=650))


# **Step 10: Video Inference & Inline Playback**
Runs license plate detection on a vehicle video and displays the processed video inline.


In [ ]:
import os
import glob
from IPython.display import HTML, display
from base64 import b64encode
from ultralytics import YOLOv10

# Download a sample car video
sample_video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4"
sample_video = "/content/yolov10/sample_traffic.mp4"

if not os.path.exists(sample_video):
    import urllib.request
    print("Downloading sample traffic video...")
    urllib.request.urlretrieve(sample_video_url, sample_video)

# Run prediction on the video using Python API
model_eval = YOLOv10('/content/yolov10/best.pt')
model_eval.predict(
    source=sample_video,
    conf=0.25,
    save=True,
    name='predict_video',
    exist_ok=True
)

# Find predicted output video
vid_dir = '/content/yolov10/runs/detect/predict_video'
vid_files = glob.glob(os.path.join(vid_dir, '*.avi')) + glob.glob(os.path.join(vid_dir, '*.mp4'))

if vid_files:
    raw_video = vid_files[0]
    web_video = "/content/yolov10/result_web.mp4"

    # Convert to web-compatible MP4 using ffmpeg (-y ensures overwrite)
    os.system(f'ffmpeg -y -i "{raw_video}" -vcodec libx264 -crf 24 "{web_video}" > /dev/null 2>&1')

    if os.path.exists(web_video):
        mp4_bytes = open(web_video, 'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
        display(HTML(f"""
        <h4>Detection Output Video:</h4>
        <video width=600 controls autoplay loop>
            <source src="{data_url}" type="video/mp4">
        </video>
        """))
else:
    print("Prediction output video not found.")
